<a href="https://colab.research.google.com/github/itsrealfarman/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsrealfarman/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Turning the validated ML-09 model into a practical, human-reviewed playbook. Everything here builds on the same March 2026 slice, the same five features, and the same client-grouped validation used since ML-04.


In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, pandas as pd, numpy as np, json as jsonlib, os

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

MONTH_START = "2026-03-01"
MID_MONTH   = "2026-03-16"
MONTH_END   = "2026-04-01"

print("Connected.")


Connected.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Model:** the same client-grouped Random Forest validated in ML-09 (Precision@50 = 0.56 on the honest split), now retrained on the full labeled slice for the playbook itself — the validation work is done, this is the deployment queue.

**Archetype → action mapping**, combining position tier (from ML-05) with the model's decline probability and a new rule that directly answers a limit found in ML-06/ML-09: the model tends to flag already-bottomed-out pages (very poor position, near-zero CTR) as "declining" when they're really just chronically weak. Those get routed to `monitor_only`, not a rewrite action — acting on a page that has nowhere lower to fall wastes the review budget the same way the ML-05 signal checks warned about.

| Archetype | Reason code | Action |
|---|---|---|
| top_10/top_20, high decline probability, real CTR gap vs tier | `ctr_below_tier_expectation` | `review_title_meta` |
| top_10/top_20, high decline probability, no clear CTR gap | `broad_decline_signal` | `refresh_content` |
| any tier, high decline probability, but already at floor CTR/position | `chronically_weak_not_declining` | `monitor_only` |
| page_1_top3, low decline probability | `stable_top_performer` | `no_action` |
| below volume floor | `insufficient_volume` | `no_action` |


In [2]:
from sklearn.ensemble import RandomForestClassifier

honest_features = ["avg_impressions_h1", "avg_clicks_h1", "ctr_h1", "avg_position_h1", "days_with_impressions_h1"]

features = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        AVG(gsc_impressions)                                    AS avg_impressions_h1,
        AVG(gsc_clicks)                                         AS avg_clicks_h1,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)       AS ctr_h1,
        AVG(gsc_avg_position)                                   AS avg_position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions_h1
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MID_MONTH}'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 500
""").df()

labels = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_h2
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MID_MONTH}' AND report_date < DATE '{MONTH_END}'
    GROUP BY 1, 2
""").df()

data = features.merge(labels, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining"] = (data["imp_h2"] < 0.8 * data["avg_impressions_h1"] * 15).astype(int)
data = data.dropna()

def position_tier(p):
    if p <= 3:  return "page_1_top3"
    if p <= 10: return "top_10"
    if p <= 20: return "top_20"
    return "beyond_20"

data["position_tier"] = data["avg_position_h1"].apply(position_tier)

# Final playbook model: retrained on the full labeled slice (validation already done in ML-09)
final_model = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
final_model.fit(data[honest_features], data["is_declining"])
data["decline_probability"] = final_model.predict_proba(data[honest_features])[:, 1]

# expected CTR per tier, for the ctr-gap reason code
expected_ctr = data.groupby("position_tier")["ctr_h1"].mean().to_dict()
data["expected_ctr_tier"] = data["position_tier"].map(expected_ctr)
data["ctr_gap"] = (data["expected_ctr_tier"] - data["ctr_h1"]).clip(lower=0)

HIGH_PROB = 0.60
CHRONIC_POSITION = 30   # beyond this, "declining further" is mostly noise on an already-weak page
CHRONIC_CTR = 0.001     # near-zero CTR already — nowhere lower to meaningfully fall

def assign_reason_action(row):
    if row["decline_probability"] < HIGH_PROB:
        if row["position_tier"] == "page_1_top3":
            return "stable_top_performer", "no_action"
        return "low_decline_probability", "no_action"
    # high decline probability from here down
    if row["avg_position_h1"] >= CHRONIC_POSITION and row["ctr_h1"] <= CHRONIC_CTR:
        return "chronically_weak_not_declining", "monitor_only"
    if row["position_tier"] in ("top_10", "top_20") and row["ctr_gap"] > 0.001:
        return "ctr_below_tier_expectation", "review_title_meta"
    return "broad_decline_signal", "refresh_content"

data[["reason_code", "action"]] = data.apply(lambda r: pd.Series(assign_reason_action(r)), axis=1)

queue = data.sort_values("decline_probability", ascending=False).reset_index(drop=True)
print(queue["action"].value_counts())
print()
queue[["content_hash_id", "position_tier", "decline_probability", "reason_code", "action"]].head(10)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

action
no_action            31209
review_title_meta     5400
refresh_content       3246
monitor_only          1961
Name: count, dtype: int64



,content_hash_id,position_tier,decline_probability,reason_code,action
0,content_54488b3c9ca74731,beyond_20,0.914361,chronically_weak_not_declining,monitor_only
1,content_877c08cf2d1997a1,beyond_20,0.913117,chronically_weak_not_declining,monitor_only
2,content_3ecbd0e3911c0a35,beyond_20,0.912692,chronically_weak_not_declining,monitor_only
3,content_58cbfe1eec8a881e,beyond_20,0.912418,chronically_weak_not_declining,monitor_only
4,content_87b9c790d43001dc,beyond_20,0.912202,chronically_weak_not_declining,monitor_only
5,content_2f094ec88d7faa51,beyond_20,0.911282,chronically_weak_not_declining,monitor_only
6,content_e70886c63f95aa1c,beyond_20,0.909362,chronically_weak_not_declining,monitor_only
7,content_0b5d0ab50b14a4e0,beyond_20,0.908886,chronically_weak_not_declining,monitor_only
8,content_ee2230ea13e45ef4,beyond_20,0.908322,chronically_weak_not_declining,monitor_only
9,content_a12d89af10c7513a,beyond_20,0.907789,chronically_weak_not_declining,monitor_only


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who:** a FlyRank content reviewer running a weekly triage pass, working from the top of the ranked queue down, choosing which pages to open first.

**What it's for:** deciding *review order*, not deciding outcomes. The queue tells a reviewer where to look first — it never fixes a page by itself, and a `review_title_meta` row is a suggestion to inspect, not an instruction to auto-publish.

**Where it stops being valid:**
- **Time:** trained and evaluated on one month (March 2026). Confidence in the ranking decays the further a scoring run gets from that window — this needs re-validation before being trusted on, say, a July 2026 slice.
- **Volume floor:** pages under 500 impressions in the scoring window are marked `insufficient_volume` and excluded — the model was never validated on low-traffic pages, and ML-04 showed exactly how noisy percentage-based labels get down there.
- **New clients:** ML-09's grouped test showed 0.56 Precision@50 on clients never seen in training — a real number, but it's an average across many clients; any single new client could sit well above or below it, especially one whose content style differs a lot from the training set.
- **Causality:** per the ML-09 paper audit, nothing here proves that *acting* on a flagged page (e.g. rewriting a title) causes recovery — the queue ranks candidates worth a human's attention, it does not claim the fix will work.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any `review_title_meta` or `refresh_content` row, a human should check:**
- Is a title/meta change already in progress for this page? (would make the flag stale)
- Is this page's low CTR intentional — a snippet written to filter low-intent clicks on purpose?
- Is there a seasonal explanation for a temporary dip that will reverse on its own?
- Does this page belong to a client with very little training history (check `dim_clients.gsc_data_start`) — treat those scores with extra caution.

**Never automate, under any version of this pipeline:**
- **Auto-publishing rewritten titles/metadata** without a human sign-off — the model ranks candidates, it does not write or approve copy.
- **Auto-merging or redirecting pages** flagged as overlapping/cannibalizing — that's a structural site decision with SEO risk far beyond what this model was trained to judge.
- **Treating a high `decline_probability` as proof of cause** — per the ML-09 audit of the paper's own Freshness Multiplier finding, correlation here is not evidence that a specific fix will work.
- **Scoring pages from clients or content types absent from the training data** without first re-validating — the client-grouped test result (0.56) is an average, not a guarantee for any one new client.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Precision@50 drift:** re-score a fresh month's held-out client-grouped slice monthly. If Precision@50 drops toward the ML-05 baseline's 0.28 (i.e., the model's edge over the simple rule disappears), that's a retrain trigger.
- **Feature distribution drift:** track the monthly mean of `ctr_h1` and `avg_impressions_h1` across the scored population. A large shift (e.g., a site-wide algorithm update changing baseline CTR) means the "expected CTR per tier" reference the reason codes rely on is now stale and needs recomputing.
- **Declining-rate shift:** if the overall `is_declining` rate moves sharply from the ~50% seen in March 2026, the underlying content mix or market conditions may have changed enough to warrant a fresh look at the label threshold itself, not just a retrain.
- **New-client cadence:** any client with less than one full month of history should be excluded from scoring until they cross that bar — scoring them earlier repeats the label-noise problem ML-04 surfaced at low volume.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Three exports: the ranked queue CSV (stays out of git — regenerated on every run), a metrics JSON (committed — the receipts my paper's numbers trace back to), and one figure (committed to `work/figures/`).


In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1. Ranked queue CSV (git-ignored by design, regenerated on every run)
queue.to_csv("work/outputs/action_playbook_queue.csv", index=False)

# 2. Metrics JSON — committed, these are the receipts
metrics = {
    "model": "RandomForestClassifier (client-grouped, ML-09 validated)",
    "validated_precision_at_50": {
        "baseline_ML05": 0.28,
        "model_naive_split_ML09": 0.82,
        "model_grouped_split_ML09": 0.56,
    },
    "playbook_month": f"{MONTH_START} to {MONTH_END}",
    "feature_window": f"{MONTH_START} to {MID_MONTH}",
    "label_window": f"{MID_MONTH} to {MONTH_END}",
    "volume_floor_impressions": 500,
    "queue_size": int(len(queue)),
    "action_counts": queue["action"].value_counts().to_dict(),
}
with open("work/outputs/playbook_metrics.json", "w") as f:
    jsonlib.dump(metrics, f, indent=2)

print("Wrote work/outputs/action_playbook_queue.csv and work/outputs/playbook_metrics.json")
print()
print(jsonlib.dumps(metrics, indent=2))


Wrote work/outputs/action_playbook_queue.csv and work/outputs/playbook_metrics.json

{
  "model": "RandomForestClassifier (client-grouped, ML-09 validated)",
  "validated_precision_at_50": {
    "baseline_ML05": 0.28,
    "model_naive_split_ML09": 0.82,
    "model_grouped_split_ML09": 0.56
  },
  "playbook_month": "2026-03-01 to 2026-04-01",
  "feature_window": "2026-03-01 to 2026-03-16",
  "label_window": "2026-03-16 to 2026-04-01",
  "volume_floor_impressions": 500,
  "queue_size": 41816,
  "action_counts": {
    "no_action": 31209,
    "review_title_meta": 5400,
    "refresh_content": 3246,
    "monitor_only": 1961
  }
}


In [7]:
# 3. One figure: queue composition by action, for the paper
action_counts = queue["action"].value_counts()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(action_counts.index, action_counts.values, color="#2563EB")
ax.set_ylabel("Pages")
ax.set_title("Content Action Playbook — Queue Composition")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("work/figures/playbook_action_counts.png", dpi=150)
plt.show()
print("Saved work/figures/playbook_action_counts.png")


Saved work/figures/playbook_action_counts.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Ranked queue with reason codes and an archetype→action mapping
- [ ] Intended use, limits, human-review rules, and a no-go list are all explicit
- [ ] Monitoring/retrain triggers are concrete, not vague
- [ ] Queue CSV, metrics JSON, and at least one figure exported from the notebook
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` (and `work/figures/`, `work/outputs/*.json`) — then submit your repo URL on the card. Done.
